# EViT-ToMe Hybrid Token Reduction Testing & Visualization (ImageNet-100)

This notebook tests the **EViT-then-ToMe hybrid** token reduction strategy and provides visualizations of:
- Which patches are kept vs. fused by EViT at each reduction layer
- Which tokens are merged by ToMe after EViT fusion
- Combined two-stage token reduction statistics through the network
- Performance vs. accuracy trade-offs
- Ablations: EViT-only, ToMe-only, and Hybrid at matched total budgets
- Comparison across keep-rate schedules and reduction locations

**Hybrid Strategy**: EViT first scores patches by CLS-attention and fuses low-attention tokens into one extra token; ToMe then merges similar surviving token pairs via bipartite soft-matching. Both stages run within the same block from a single QKV computation, controlled by independent keep-rate schedules.

In [ ]:
# If running in a fresh Colab runtime, uncomment these lines:
# !git clone https://github.com/Chalhotra/ViT-Token-Economy.git
# %cd ViT-Token-Economy

In [ ]:
!git checkout test-branch

In [ ]:
!pip -q install -r requirements.txt
!pip -q install -e .

In [ ]:
# Import core modules
from src.imagenet_mapping import build_imagenet100_to_1k_map
from src.models import ModelConfig, create_model, shrink_imagenet1k_head_to_imagenet100
from src.data import DataConfig, load_imagenet100_split, build_transform_for_model, apply_timm_preprocess, build_loader
from src.eval import evaluate_accuracy_latency_throughput, compute_gflops
from src.utils import get_device, num_params
from src.test_models.evit_tome import (
    EVITToMeConfig, apply_evit_tome_pruning,
    BlockHybridAdapter,
)
# Individual methods for ablation baselines
from src.test_models.evit import EVITConfig, apply_evit_pruning
from src.test_models.tome import ToMeConfig, apply_tome_merging
import torch

# Visualization
import matplotlib.pyplot as plt  # pyright: ignore[reportMissingModuleSource]
import numpy as np
import seaborn as sns # pyright: ignore[reportMissingModuleSource]
from typing import List, Dict
import pandas as pd

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
device = get_device()
maps = build_imagenet100_to_1k_map()
print(f"Using device: {device}")

## Utility Functions for Hybrid Visualization

In [ ]:
def extract_hybrid_info(model, x):
    """
    Walk through blocks and collect per-layer token counts and reduction events.
    Returns:
        evit_events : list of (block_idx, kept_indices [B, K], compl_indices [B, P])
        tome_events : list of (block_idx, cluster_idx [B, T_merged])
        token_counts: list of patch-token counts after each block (excl. special tokens)
    """
    evit_events, tome_events, token_counts = [], [], []
    B, N, C = x.shape
    num_special = 2 if hasattr(model, 'dist_token') and model.dist_token is not None else 1
    token_counts.append(N - num_special)

    for i, block in enumerate(model.blocks):
        with torch.no_grad():
            x = block(x)

        if isinstance(block, BlockHybridAdapter):
            # EViT event
            if block.last_evit_idx is not None:
                kept = block.last_evit_idx[:, :-1]   # strip -1 sentinel
                compl = block.last_evit_compl
                evit_events.append((i, kept.cpu(), compl.cpu() if compl is not None else None))
            # ToMe event
            if block.last_tome_cluster is not None:
                tome_events.append((i, block.last_tome_cluster.cpu()))

        token_counts.append(x.shape[1] - num_special)

    return evit_events, tome_events, token_counts


def _patch_overlay(img_np, patch_size, mask, color, alpha=0.55):
    """Blend a solid color over patches where mask==True."""
    overlay = img_np.copy()
    n_h, n_w = img_np.shape[0] // patch_size, img_np.shape[1] // patch_size
    for flat_idx in range(n_h * n_w):
        if not mask[flat_idx]:
            continue
        r, c_ = flat_idx % n_h, flat_idx // n_h
        ys, ye = r * patch_size, (r + 1) * patch_size
        xs, xe = c_ * patch_size, (c_ + 1) * patch_size
        for ch, cv in enumerate(color):
            overlay[ys:ye, xs:xe, ch] = np.clip(
                overlay[ys:ye, xs:xe, ch] * (1 - alpha) + cv * alpha, 0, 1
            )
    return overlay


def visualize_hybrid_reduction(image, patch_size, evit_events, tome_events, token_counts, model_name):
    """
    Side-by-side panels for each active block:
      - EViT panel: orange tint = fused patches, rest = kept
      - ToMe panel: coloured borders per merged-token cluster
    """
    if isinstance(image, torch.Tensor):
        img_np = image.permute(1, 2, 0).cpu().numpy()
    else:
        img_np = np.array(image).astype(float) / 255.0 if np.array(image).max() > 1 else np.array(image, dtype=float)

    h, w = img_np.shape[:2]
    n_ph, n_pw = h // patch_size, w // patch_size
    total_patches = n_ph * n_pw

    # Build a combined ordered event list for panels
    all_events = []
    evit_dict = {idx: (kept, compl) for idx, kept, compl in evit_events}
    tome_dict  = {idx: cluster for idx, cluster in tome_events}
    all_block_ids = sorted(set(list(evit_dict.keys()) + list(tome_dict.keys())))

    n_panels = 1 + len(all_block_ids)
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))
    if n_panels == 1:
        axes = [axes]

    axes[0].imshow(img_np)
    axes[0].set_title(f'Original\n{total_patches} patches')
    axes[0].axis('off')

    for panel_i, blk_idx in enumerate(all_block_ids):
        ax = axes[panel_i + 1]
        base = img_np.copy()
        title_lines = [f'After block {blk_idx}']

        # EViT layer: orange tint fused patches
        if blk_idx in evit_dict:
            kept_idx, compl_idx = evit_dict[blk_idx]
            fused_mask = np.zeros(total_patches, bool)
            if compl_idx is not None:
                for fi in compl_idx[0].numpy():
                    if 0 <= fi < total_patches:
                        fused_mask[fi] = True
            base = _patch_overlay(base, patch_size, fused_mask, (1.0, 0.45, 0.0))
            kept_n = kept_idx.shape[1]
            fused_n = int(fused_mask.sum())
            title_lines.append(f'EViT: {kept_n} kept + 1 fused ({fused_n} → 1)')

        # ToMe layer: coloured patch borders per cluster
        if blk_idx in tome_dict:
            clusters_np = tome_dict[blk_idx][0].numpy()
            n_clust = len(clusters_np)
            cmap = plt.cm.get_cmap('tab20' if n_clust <= 20 else 'hsv')
            for ci, cluster_val in enumerate(clusters_np):
                pidx = int(cluster_val)
                if 0 <= pidx < total_patches:
                    color = cmap(ci / max(1, n_clust - 1))[:3]
                    r, c_ = pidx % n_ph, pidx // n_ph
                    ys, ye = r * patch_size, (r + 1) * patch_size
                    xs, xe = c_ * patch_size, (c_ + 1) * patch_size
                    bw = 2
                    base[ys:ys+bw, xs:xe] = color
                    base[ye-bw:ye, xs:xe] = color
                    base[ys:ye, xs:xs+bw] = color
                    base[ys:ye, xe-bw:xe] = color
            merged_away = total_patches - n_clust
            title_lines.append(f'ToMe: {n_clust} tokens ({merged_away} merged)')

        ax.imshow(np.clip(base, 0, 1))
        ax.set_title('\n'.join(title_lines))
        ax.axis('off')

    plt.suptitle(f'{model_name} — Hybrid Reduction Visualization', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


def plot_hybrid_token_timeline(token_counts, evit_locs, tome_locs, model_name):
    """Token count through layers, annotating EViT and ToMe reduction points."""
    layers = list(range(len(token_counts)))
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(layers, token_counts, marker='o', linewidth=2, markersize=8,
            label='Patch tokens (excl. special)', color='steelblue')

    for loc in evit_locs:
        if loc < len(token_counts):
            ax.axvline(x=loc, color='darkorange', linestyle='--', alpha=0.6)
            ax.text(loc + 0.1, max(token_counts) * 0.97, f'EViT@{loc}',
                    color='darkorange', rotation=90, va='top', fontsize=8)
    for loc in tome_locs:
        if loc < len(token_counts):
            ax.axvline(x=loc, color='mediumseagreen', linestyle=':', alpha=0.6)
            ax.text(loc + 0.1, max(token_counts) * 0.88, f'ToMe@{loc}',
                    color='mediumseagreen', rotation=90, va='top', fontsize=8)

    initial = token_counts[0]
    key_locs = set(evit_locs) | set(tome_locs) | {0, len(token_counts) - 1}
    for i, cnt in enumerate(token_counts):
        if i in key_locs:
            pct = cnt / initial * 100
            ax.annotate(f'{cnt} ({pct:.1f}%)',
                        xy=(i, cnt), xytext=(0, 12), textcoords='offset points',
                        ha='center', fontsize=8,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.7))

    ax.set_xlabel('Layer Index', fontsize=12)
    ax.set_ylabel('Number of Patch Tokens', fontsize=12)
    ax.set_title(f'{model_name} — Token Count Through Layers (Hybrid)', fontsize=14)
    from matplotlib.lines import Line2D
    legend_items = [
        Line2D([0],[0], color='steelblue', marker='o', label='Token count'),
        Line2D([0],[0], color='darkorange', linestyle='--', label='EViT fusion'),
        Line2D([0],[0], color='mediumseagreen', linestyle=':', label='ToMe merge'),
    ]
    ax.legend(handles=legend_items)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def compare_configurations(results_list: List[Dict], title: str = "Configuration Comparison"):
    """4-panel comparison plot identical in style to the EViT/ToMe notebooks."""
    df = pd.DataFrame(results_list)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].bar(range(len(df)), df['acc1'], color='steelblue')
    axes[0, 0].set_xticks(range(len(df)))
    axes[0, 0].set_xticklabels(df['config_name'], rotation=45, ha='right')
    axes[0, 0].set_ylabel('Top-1 Accuracy (%)')
    axes[0, 0].set_title('Accuracy Comparison')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].bar(range(len(df)), df['gflops'], color='coral')
    axes[0, 1].set_xticks(range(len(df)))
    axes[0, 1].set_xticklabels(df['config_name'], rotation=45, ha='right')
    axes[0, 1].set_ylabel('GFLOPs')
    axes[0, 1].set_title('Computational Cost')
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].bar(range(len(df)), df['latency_ms'], color='mediumseagreen')
    axes[1, 0].set_xticks(range(len(df)))
    axes[1, 0].set_xticklabels(df['config_name'], rotation=45, ha='right')
    axes[1, 0].set_ylabel('Latency (ms)')
    axes[1, 0].set_title('Inference Latency')
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].scatter(df['gflops'], df['acc1'], s=100, alpha=0.7,
                       c=range(len(df)), cmap='viridis')
    for _, row in df.iterrows():
        axes[1, 1].annotate(row['config_name'], (row['gflops'], row['acc1']),
                            xytext=(5, 5), textcoords='offset points', fontsize=8)
    axes[1, 1].set_xlabel('GFLOPs')
    axes[1, 1].set_ylabel('Top-1 Accuracy (%)')
    axes[1, 1].set_title('Efficiency Plot')
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

    print("\n" + "="*80)
    print(f"{title} — Summary Table")
    print("="*80)
    print(df.to_string(index=False))
    print("="*80 + "\n")

In [ ]:
def plot_hybrid_results_spacious(results_list: List[Dict], title: str = "Hybrid Cartesian Sweep Results"):
    """
    Spacious 2x2 comparison plot for hybrid sweep results with extra padding.
    Panels:
      1) Top-1 accuracy by config
      2) GFLOPs by config
      3) Latency by config
      4) Accuracy vs GFLOPs efficiency scatter
    """
    if not results_list:
        print("No results to plot.")
        return

    df = pd.DataFrame(results_list).copy()
    if df.empty:
        print("No results to plot.")
        return

    # Keep only hybrid rows if present; otherwise plot everything.
    hybrid_df = df[df["config_name"].str.startswith("Hybrid EViT=", na=False)].copy()
    plot_df = hybrid_df if not hybrid_df.empty else df.copy()

    # Parse and sort Cartesian names for stable visual ordering.
    if "config_name" in plot_df.columns and plot_df["config_name"].str.contains("Hybrid EViT=", na=False).any():
        parsed = plot_df["config_name"].str.extract(r"Hybrid EViT=(\d+\.\d+) ToMe=(\d+\.\d+)")
        plot_df["evit_r"] = pd.to_numeric(parsed[0], errors="coerce")
        plot_df["tome_r"] = pd.to_numeric(parsed[1], errors="coerce")
        plot_df = plot_df.sort_values(["evit_r", "tome_r", "config_name"], na_position="last")

    n = len(plot_df)
    x = np.arange(n)

    # Build compact but readable x labels.
    if {"evit_r", "tome_r"}.issubset(plot_df.columns):
        xlabels = [f"E{er:.2f}|T{tr:.2f}" for er, tr in zip(plot_df["evit_r"], plot_df["tome_r"])]
    else:
        xlabels = plot_df["config_name"].astype(str).tolist()

    plt.close("all")
    fig, axes = plt.subplots(
        2, 2, figsize=(24, 16), dpi=130,
        constrained_layout=False
    )

    # Large paddings to avoid any overlap in dense labels/annotations.
    fig.subplots_adjust(
        left=0.06, right=0.985, bottom=0.24, top=0.90,
        wspace=0.28, hspace=0.42
    )

    # Panel 1: Accuracy
    ax = axes[0, 0]
    ax.bar(x, plot_df["acc1"], color="steelblue", alpha=0.9)
    ax.set_title("Top-1 Accuracy", fontsize=16, pad=16)
    ax.set_ylabel("Accuracy (%)", fontsize=13, labelpad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=45, ha="right", fontsize=10)
    ax.tick_params(axis="x", pad=8)
    ax.tick_params(axis="y", pad=6)
    ax.grid(True, axis="y", alpha=0.30)

    # Panel 2: GFLOPs
    ax = axes[0, 1]
    ax.bar(x, plot_df["gflops"], color="coral", alpha=0.9)
    ax.set_title("Computational Cost", fontsize=16, pad=16)
    ax.set_ylabel("GFLOPs", fontsize=13, labelpad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=45, ha="right", fontsize=10)
    ax.tick_params(axis="x", pad=8)
    ax.tick_params(axis="y", pad=6)
    ax.grid(True, axis="y", alpha=0.30)

    # Panel 3: Latency
    ax = axes[1, 0]
    ax.bar(x, plot_df["latency_ms"], color="mediumseagreen", alpha=0.9)
    ax.set_title("Inference Latency", fontsize=16, pad=16)
    ax.set_ylabel("Latency (ms)", fontsize=13, labelpad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=45, ha="right", fontsize=10)
    ax.tick_params(axis="x", pad=8)
    ax.tick_params(axis="y", pad=6)
    ax.grid(True, axis="y", alpha=0.30)

    # Panel 4: Efficiency
    ax = axes[1, 1]
    sc = ax.scatter(
        plot_df["gflops"], plot_df["acc1"],
        s=150, alpha=0.85, c=np.arange(n), cmap="viridis",
        edgecolors="black", linewidths=0.4
    )
    ax.set_title("Efficiency: Accuracy vs GFLOPs", fontsize=16, pad=16)
    ax.set_xlabel("GFLOPs", fontsize=13, labelpad=10)
    ax.set_ylabel("Top-1 Accuracy (%)", fontsize=13, labelpad=10)
    ax.tick_params(axis="both", pad=6)
    ax.grid(True, alpha=0.30)

    # Annotate with short labels and point offsets to reduce collisions.
    for i, (_, row) in enumerate(plot_df.iterrows()):
        label = xlabels[i] if i < len(xlabels) else str(row.get("config_name", i))
        xoff = 8 if i % 2 == 0 else -8
        yoff = 8 if i % 3 else -10
        ax.annotate(
            label,
            (row["gflops"], row["acc1"]),
            xytext=(xoff, yoff),
            textcoords="offset points",
            fontsize=9,
            ha="left" if xoff > 0 else "right",
            va="bottom" if yoff > 0 else "top",
            bbox=dict(boxstyle="round,pad=0.22", facecolor="white", alpha=0.72, edgecolor="none")
        )

    # Shared colorbar with extra padding.
    cbar = fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.82, pad=0.02)
    cbar.set_label("Configuration Index", fontsize=11, labelpad=8)

    fig.suptitle(title, fontsize=20, y=0.965)
    plt.show()

## Core Test Runner

In [ ]:
def run_hybrid_test(
    model_id: str,
    cfg: EVITToMeConfig,
    config_name: str,
    batch_size: int = 64,
    visualize: bool = True,
):
    """
    Run model with a given EVITToMeConfig, evaluate, and optionally visualize.
    Also accepts plain EVITConfig / ToMeConfig for ablation baselines.
    """
    print(f"\n{'='*80}")
    print(f"Testing: {config_name}")
    print(f"Model: {model_id}")
    if isinstance(cfg, EVITToMeConfig):
        print(f"EViT locs : {cfg.evit_reduction_loc}  keep_rate: {cfg.evit_keep_rate}")
        print(f"ToMe locs : {cfg.tome_reduction_loc}  keep_rate: {cfg.tome_keep_rate}")
        print(f"prop_attn : {cfg.tome_prop_attn}")
    elif isinstance(cfg, EVITConfig):
        print(f"[EViT-only] locs: {cfg.reduction_loc}  keep_rate: {cfg.keep_rate}")
    elif isinstance(cfg, ToMeConfig):
        print(f"[ToMe-only] locs: {cfg.reduction_loc}  keep_rate: {cfg.keep_rate}")
    print(f"{'='*80}\n")

    model = create_model(ModelConfig(model_id=model_id, pretrained=True))
    model = shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)

    if isinstance(cfg, EVITToMeConfig):
        model = apply_evit_tome_pruning(model, cfg)
    elif isinstance(cfg, EVITConfig):
        model = apply_evit_pruning(model, cfg)
    elif isinstance(cfg, ToMeConfig):
        model = apply_tome_merging(model, cfg)

    model = model.to(device).eval()

    ds        = load_imagenet100_split(DataConfig(split='validation'))
    transform = build_transform_for_model(model)
    ds_t      = apply_timm_preprocess(ds, transform)
    loader    = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False))

    metrics = evaluate_accuracy_latency_throughput(model, loader, device)
    sample  = ds_t[0]['pixel_values'].unsqueeze(0).to(device)
    gflops  = compute_gflops(model, sample)

    if visualize and isinstance(cfg, EVITToMeConfig) and (cfg.evit_enabled or cfg.tome_enabled):
        sample_idx    = 42
        sample_image  = ds[sample_idx]['image']
        sample_tensor = ds_t[sample_idx]['pixel_values'].unsqueeze(0).to(device)

        with torch.no_grad():
            x = model.patch_embed(sample_tensor)
            if hasattr(model, 'cls_token'):
                x = torch.cat((model.cls_token.expand(1, -1, -1), x), dim=1)
            if hasattr(model, 'pos_embed'):
                x = x + model.pos_embed
            if hasattr(model, 'pos_drop'):
                x = model.pos_drop(x)
            evit_events, tome_events, token_counts = extract_hybrid_info(model, x)

        if evit_events or tome_events:
            visualize_hybrid_reduction(
                sample_image, 16,
                evit_events, tome_events,
                token_counts, f"{model_id} — {config_name}"
            )
            plot_hybrid_token_timeline(
                token_counts,
                list(cfg.evit_reduction_loc),
                list(cfg.tome_reduction_loc),
                f"{model_id} — {config_name}"
            )

    result = {
        'config_name': config_name,
        'model': model_id,
        'params_m': num_params(model) / 1e6,
        'gflops': gflops,
        **metrics
    }

    print(f"\nResults for {config_name}:")
    print(f"  Top-1 Accuracy : {metrics['acc1']:.2f}%")
    print(f"  GFLOPs         : {gflops:.3f}")
    print(f"  Latency        : {metrics['latency_ms']:.2f} ms")
    print(f"  Throughput     : {metrics['throughput']:.1f} samples/sec")
    return result

## Baseline (No Reduction)

In [ ]:
baseline_result = run_hybrid_test(
    model_id='deit_tiny_patch16_224',
    cfg=EVITToMeConfig(evit_enabled=False, tome_enabled=False),
    config_name='Baseline (No Reduction)',
    visualize=False
)
results = [baseline_result]

## Sanity Check: Keep Rate = 1.0 (Should Match Baseline)

In [ ]:
# Both stages enabled but keep_rate = 1.0 everywhere → no actual reduction
sanity_cfg = EVITToMeConfig(
    evit_enabled=True,
    evit_keep_rate=(1.0,),
    evit_reduction_loc=(3, 6, 9),
    evit_exponentiate=False,
    tome_enabled=True,
    tome_keep_rate=(1.0,),
    tome_reduction_loc=(3, 6, 9),
    tome_exponentiate=False,
)
sanity_result = run_hybrid_test(
    model_id='deit_tiny_patch16_224',
    cfg=sanity_cfg,
    config_name='Sanity Check (keep=1.0)',
    visualize=True
)
results.append(sanity_result)

In [ ]:
# Cartesian hybrid sweep: EViT keep-rate x ToMe keep-rate (4 x 4 = 16 tests)
rates = [0.25, 0.50, 0.70, 0.90]

if 'results' not in globals():
    results = [baseline_result, sanity_result]

for evit_r in rates:
    for tome_r in rates:
        cfg = EVITToMeConfig(
            evit_enabled=True,
            evit_keep_rate=(evit_r, evit_r, evit_r),
            evit_reduction_loc=(3, 6, 9),
            evit_exponentiate=False,
            tome_enabled=True,
            tome_keep_rate=(tome_r, tome_r, tome_r),
            tome_reduction_loc=(3, 6, 9),
            tome_exponentiate=False,
            tome_prop_attn=True,
            viz_mode=False,
        )
        name = f"Hybrid EViT={evit_r:.2f} ToMe={tome_r:.2f}"

        if not any(x.get('config_name') == name for x in results):
            out = run_hybrid_test(
                model_id='deit_tiny_patch16_224',
                cfg=cfg,
                config_name=name,
                visualize=False,
            )
            results.append(out)

In [ ]:
# Accuracy/GFLOPs analysis (separate cell): rank configs and report best
if 'df_cart' in globals() and isinstance(df_cart, pd.DataFrame) and not df_cart.empty:
    analysis_df = df_cart.copy()
else:
    analysis_df = pd.DataFrame(results)
    analysis_df = analysis_df[analysis_df['config_name'].str.startswith('Hybrid EViT=', na=False)].copy()
    if not analysis_df.empty:
        parsed = analysis_df['config_name'].str.extract(r'Hybrid EViT=(\d+\.\d+) ToMe=(\d+\.\d+)')
        analysis_df['evit_r'] = parsed[0].astype(float)
        analysis_df['tome_r'] = parsed[1].astype(float)
        analysis_df = analysis_df.sort_values(['evit_r', 'tome_r'])

if analysis_df.empty:
    print('No Cartesian sweep results found yet. Run the Cartesian sweep cell first.')
else:
    analysis_df['acc_per_gflop'] = analysis_df['acc1'] / analysis_df['gflops']

    rank_cols = [
        c for c in [
            'config_name', 'evit_r', 'tome_r',
            'acc1', 'gflops', 'acc_per_gflop', 'latency_ms', 'throughput'
        ] if c in analysis_df.columns
    ]
    df_rank = analysis_df.sort_values('acc_per_gflop', ascending=False)[rank_cols].reset_index(drop=True)
    display(df_rank)

    best = df_rank.iloc[0]
    print('\n' + '=' * 90)
    print('Best config by Accuracy/GFLOPs')
    print('=' * 90)
    print(f"Config      : {best['config_name']}")
    print(f"EViT keep   : {best['evit_r']:.2f}")
    print(f"ToMe keep   : {best['tome_r']:.2f}")
    print(f"Top-1 Acc   : {best['acc1']:.2f}%")
    print(f"GFLOPs      : {best['gflops']:.3f}")
    print(f"Acc/GFLOP   : {best['acc_per_gflop']:.3f}")
    print(f"Latency (ms): {best['latency_ms']:.2f}")
    print(f"Throughput  : {best['throughput']:.1f} samples/sec")
    print('=' * 90 + '\n')

In [ ]:
# Accuracy/GFLOPs ranking graph (separate cell)
if 'df_rank' in globals() and isinstance(df_rank, pd.DataFrame) and not df_rank.empty:
    rank_plot_df = df_rank.copy()
else:
    rank_plot_df = pd.DataFrame(results)
    rank_plot_df = rank_plot_df[rank_plot_df['config_name'].str.startswith('Hybrid EViT=', na=False)].copy()
    if not rank_plot_df.empty:
        parsed = rank_plot_df['config_name'].str.extract(r'Hybrid EViT=(\d+\.\d+) ToMe=(\d+\.\d+)')
        rank_plot_df['evit_r'] = parsed[0].astype(float)
        rank_plot_df['tome_r'] = parsed[1].astype(float)
        rank_plot_df['acc_per_gflop'] = rank_plot_df['acc1'] / rank_plot_df['gflops']
        rank_plot_df = rank_plot_df.sort_values('acc_per_gflop', ascending=False).reset_index(drop=True)
        rank_plot_df = rank_plot_df[['config_name', 'evit_r', 'tome_r', 'acc_per_gflop', 'acc1', 'gflops']]

if rank_plot_df.empty:
    print('No Cartesian sweep results found yet. Run the sweep and analysis cells first.')
else:
    plt.close('all')
    fig, ax = plt.subplots(figsize=(16, 10), dpi=130)
    fig.subplots_adjust(left=0.32, right=0.96, top=0.90, bottom=0.08)

    labels = [f"E{er:.2f}|T{tr:.2f}" if 'evit_r' in rank_plot_df.columns and 'tome_r' in rank_plot_df.columns
              else name for er, tr, name in zip(
                  rank_plot_df.get('evit_r', pd.Series([np.nan]*len(rank_plot_df))),
                  rank_plot_df.get('tome_r', pd.Series([np.nan]*len(rank_plot_df))),
                  rank_plot_df['config_name']
              )]

    y = np.arange(len(rank_plot_df))
    values = rank_plot_df['acc_per_gflop'].astype(float).values

    bars = ax.barh(y, values, color='teal', alpha=0.88)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('Accuracy per GFLOP (higher is better)', fontsize=12, labelpad=10)
    ax.set_title('Hybrid Config Ranking by Accuracy/GFLOPs', fontsize=16, pad=14)
    ax.grid(True, axis='x', alpha=0.30)
    ax.tick_params(axis='y', pad=8)
    ax.tick_params(axis='x', pad=6)

    xmax = float(values.max()) if len(values) else 0.0
    xpad = xmax * 0.02 if xmax > 0 else 0.02
    for i, (bar, v) in enumerate(zip(bars, values)):
        acc = float(rank_plot_df.iloc[i]['acc1']) if 'acc1' in rank_plot_df.columns else np.nan
        gfl = float(rank_plot_df.iloc[i]['gflops']) if 'gflops' in rank_plot_df.columns else np.nan
        txt = f"{v:.3f}  |  acc={acc:.2f}%  gflops={gfl:.3f}"
        ax.text(v + xpad, bar.get_y() + bar.get_height() / 2, txt, va='center', fontsize=9)

    ax.set_xlim(0, xmax + xpad * 20)
    plt.show()